# HCTiGT — Rabbit Hole: Pretrain GPT-2 Yourself

**The first notebook changes an existing model. This one starts with random model weights.**

Choose a pretraining corpus, train a GPT-2 *architecture* on next-token prediction, and measure what changes. No need to implement attention, tokenization, or a model architecture yourself — [ARENA](https://arena.education/) is an excellent place for that later.

This is an **optional**, intentionally bounded experiment. It is **not** a reproduction of OpenAI's complete GPT-2 pretraining run. Even the 124M-parameter option will see only a small slice of text and a limited number of steps.

### Before you start
1. In Google Colab, choose **Runtime → Change runtime type → T4 GPU**, if available.
2. Run the setup cells.
3. **Choose your corpus and model size**, then check `RUN_PRETRAINING` and run the experiment cell.
4. Watch both *training* and *held-out* loss. Your model may still generate nonsense; this is expected.

The model weights start random, but we **reuse GPT-2's pretrained tokenizer**, so the tokenizer itself is not trained from scratch.


## 1. Set up a small training laboratory

The only model weights we'll initialize come from `GPT2LMHeadModel(config)` — **never** `from_pretrained(...)`.


In [ ]:
%pip -q install "transformers>=4.44,<5" "requests>=2.31,<3" "matplotlib>=3.7,<4"


In [ ]:
import random
from contextlib import nullcontext

import matplotlib.pyplot as plt
import requests
import torch
from transformers import AutoTokenizer, GPT2Config, GPT2LMHeadModel

SEED = 2026
random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TOKENIZER_ID = "openai-community/gpt2"
BLOCK_SIZE = 96
BATCH_SIZE = 4
print("Device:", DEVICE)
if DEVICE.type != "cuda":
    print("A GPU is recommended. The small model can run on CPU, but slowly.")


## 2. Choose a corpus

Three different diets for the same randomly initialized architecture:

- **TinyStories** — short, accessible stories ([dataset](https://huggingface.co/datasets/roneneldan/TinyStories)).
- **WikiText-2** — excerpts from Wikipedia articles ([dataset](https://huggingface.co/datasets/Salesforce/wikitext)).
- **Arithmetic** — synthetic `Question: a + b / Answer: c` text generated right in this notebook.

TinyStories and WikiText examples are fetched from the public Hugging Face dataset viewer, up to **100 rows per request**. We use their **train** and **validation** splits; no giant download. Arithmetic creates independent train and validation examples locally.

The corpus is small on purpose. It is enough to observe the *process* of pretraining, not enough to expect competent general-purpose text generation.


In [ ]:
DATASETS = {
    "TinyStories": ("roneneldan/TinyStories", "default"),
    "WikiText-2": ("Salesforce/wikitext", "wikitext-2-raw-v1"),
}


def fetch_rows(corpus, split, row_count):
    dataset, config = DATASETS[corpus]
    texts = []
    for offset in range(0, row_count, 100):
        response = requests.get(
            "https://datasets-server.huggingface.co/rows",
            params={
                "dataset": dataset,
                "config": config,
                "split": split,
                "offset": offset,
                "length": min(100, row_count - offset),
            },
            timeout=90,
        )
        response.raise_for_status()
        batch = response.json()["rows"]
        texts.extend(
            entry["row"]["text"].strip()
            for entry in batch
            if isinstance(entry["row"].get("text"), str)
            and entry["row"]["text"].strip()
        )
    if not texts:
        raise RuntimeError(f"No text returned for {corpus} / {split}.")
    return texts


def make_arithmetic(row_count, *, seed):
    rng = random.Random(seed)
    pairs = rng.sample(range(1000 * 1000), k=row_count)
    return [
        f"Question: {pair // 1000} + {pair % 1000}\nAnswer: {(pair // 1000) + (pair % 1000)}\n"
        for pair in pairs
    ]


def load_corpus(corpus, row_count):
    if corpus == "Arithmetic":
        # Sampling without replacement guarantees disjoint train/validation pairs.
        examples = make_arithmetic(row_count + 80, seed=1)
        return examples[:row_count], examples[row_count:]
    return (
        fetch_rows(corpus, "train", row_count),
        fetch_rows(corpus, "validation", 80),
    )


def pack_blocks(texts, tokenizer, block_size=BLOCK_SIZE):
    """Pack a stream of documents into fixed-length next-token training blocks."""
    tokens = []
    for text in texts:
        tokens.extend(tokenizer.encode(text, add_special_tokens=False))
        tokens.append(tokenizer.eos_token_id)
    usable = (len(tokens) // block_size) * block_size
    if usable < 2 * block_size:
        raise ValueError("Too little text for two training blocks; choose more rows.")
    return torch.tensor(tokens[:usable], dtype=torch.long).view(-1, block_size)


## 3. The architecture is given. The weights are not.

The **tiny** GPT-2 option uses four transformer blocks and roughly 16 million random parameters. It's more approachable on a free Colab GPU.

The **124M GPT-2** option uses GPT-2's original small-model configuration (12 blocks, 768-wide hidden states), but **none of its pretrained weights**. It can be much slower and use more GPU memory, and this short run will not produce a fully pretrained GPT-2.

The loss is the standard causal language-modeling objective: given some tokens, predict the next token. We'll use *held-out* validation text to guard against mistaking memorization for learning to predict unseen text.


In [ ]:
def make_random_gpt2(size, vocab_size):
    if size == "tiny GPT-2 (~16M)":
        config = GPT2Config(
            vocab_size=vocab_size,
            n_positions=128, n_ctx=128,
            n_embd=256, n_layer=4, n_head=4,
        )
    elif size == "GPT-2 small (124M)":
        config = GPT2Config(vocab_size=vocab_size)
    else:
        raise ValueError(f"Unknown model size: {size}")
    config.use_cache = False  # No key/value cache during training.
    return GPT2LMHeadModel(config)  # RANDOM initialization, not a checkpoint.


def precision_context():
    if DEVICE.type == "cuda":
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    return nullcontext()


@torch.no_grad()
def loss_on_blocks(model, blocks, max_blocks=8):
    model.eval()
    observed = []
    for batch in blocks[:max_blocks].split(BATCH_SIZE):
        with precision_context():
            loss = model(input_ids=batch.to(DEVICE), labels=batch.to(DEVICE)).loss
        observed.append((loss.item(), len(batch)))
    # Average by number of sequences; each sequence has the same length.
    return sum(loss * size for loss, size in observed) / sum(size for _, size in observed)


@torch.no_grad()
def generate_sample(model, tokenizer, corpus):
    model.eval()
    prompt = {
        "TinyStories": "Once upon a time,",
        "WikiText-2": "The history of",
        "Arithmetic": "Question: 7 + 5\nAnswer:",
    }[corpus]
    prompt_ids = tokenizer(prompt, return_tensors="pt")["input_ids"].to(DEVICE)
    torch.manual_seed(SEED + 1)
    with precision_context():
        output = model.generate(
            input_ids=prompt_ids, do_sample=True, temperature=0.9,
            max_new_tokens=48, pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output[0], skip_special_tokens=True)


def pretrain_from_scratch(corpus, model_size, row_count, steps):
    if model_size == "GPT-2 small (124M)" and DEVICE.type != "cuda":
        raise RuntimeError("The 124M model needs a GPU runtime for this exercise.")
    torch.manual_seed(SEED)
    print("Loading GPT-2 tokenizer (not model weights)...")
    tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID)

    print("Loading corpus...")
    train_texts, valid_texts = load_corpus(corpus, row_count)
    print(f"{len(train_texts)} train documents, {len(valid_texts)} validation documents.")
    print("First training sample:", repr(train_texts[0][:180]))

    train_blocks = pack_blocks(train_texts, tokenizer)
    valid_blocks = pack_blocks(valid_texts, tokenizer)
    print(f"{len(train_blocks)} train blocks, {len(valid_blocks)} validation blocks.")

    model = make_random_gpt2(model_size, tokenizer.vocab_size).to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"Randomly initialized parameters: {n_params:,}")
    original_weights = model.transformer.h[0].attn.c_attn.weight.detach().float().cpu().clone()

    first_train = loss_on_blocks(model, train_blocks)
    first_valid = loss_on_blocks(model, valid_blocks)
    print(f"Step 0: train={first_train:.3f}, held-out={first_valid:.3f}")
    initial_text = generate_sample(model, tokenizer, corpus)

    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4)
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))
    batch_rng = torch.Generator().manual_seed(SEED)
    trajectory = [(0, first_train, first_valid)]

    for step in range(1, steps + 1):
        model.train()
        chosen = torch.randint(len(train_blocks), (BATCH_SIZE,), generator=batch_rng)
        batch = train_blocks[chosen].to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with precision_context():
            loss = model(input_ids=batch, labels=batch).loss
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        if step % 20 == 0 or step == steps:
            train_loss = loss_on_blocks(model, train_blocks)
            valid_loss = loss_on_blocks(model, valid_blocks)
            trajectory.append((step, train_loss, valid_loss))
            print(f"Step {step:>3}: train={train_loss:.3f}, held-out={valid_loss:.3f}")

    final_text = generate_sample(model, tokenizer, corpus)
    change = (model.transformer.h[0].attn.c_attn.weight.detach().float().cpu() - original_weights).abs().max().item()
    print(f"Maximum change in a first-layer parameter: {change:.6g}")
    print("\nBefore pretraining:\n", initial_text)
    print("\nAfter pretraining:\n", final_text)

    steps_x, training_y, validation_y = zip(*trajectory)
    plt.figure(figsize=(7, 3.5))
    plt.plot(steps_x, training_y, label="Training probe loss")
    plt.plot(steps_x, validation_y, label="Held-out probe loss")
    plt.xlabel("Optimizer steps")
    plt.ylabel("Next-token loss (lower is better)")
    plt.title(f"From random weights: {corpus}")
    plt.legend()
    plt.show()
    print("Reminder: sampled text isn't a controlled evaluation, and this is a short pretraining run.")
    return model, tokenizer, trajectory


## 4. Choose, predict, run

Before checking the box: Which corpus do you think will reduce held-out loss fastest? Will the resulting model complete your prompt in the style you expect? Why?

The **tiny** model is the sensible first choice. The 124M option is here for the curious, not a requirement.

**Run all won't train by accident:** explicitly check `RUN_PRETRAINING` and execute the cell below once you've made a choice. Changing the controls and rerunning creates a new randomly initialized model. To keep GPU memory free, previous results are moved off the GPU.


In [ ]:
# @title 🧪 Pretrain GPT-2 from random weights { display-mode: "form" }
RUN_PRETRAINING = False  # @param {type:"boolean"}
CORPUS = "TinyStories"  # @param ["TinyStories", "WikiText-2", "Arithmetic"]
MODEL_SIZE = "tiny GPT-2 (~16M)"  # @param ["tiny GPT-2 (~16M)", "GPT-2 small (124M)"]
TRAIN_ROWS = 400  # @param {type:"slider", min:100, max:800, step:100}
STEPS = 120  # @param {type:"slider", min:20, max:400, step:20}

if not RUN_PRETRAINING:
    print("Choose your corpus and model, check RUN_PRETRAINING, then press ▶ on this cell.")
else:
    if "scratch_model" in globals():
        scratch_model.to("cpu")
        del scratch_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    scratch_model, scratch_tokenizer, scratch_trajectory = pretrain_from_scratch(
        CORPUS, MODEL_SIZE, TRAIN_ROWS, STEPS
    )


## 5. What did the model actually learn?

You've now performed **actual next-token pretraining** of a GPT-2 architecture with **randomly initialized weights**. The training corpus was *your choice*. No one taught this run to write an answer in a separate supervised fine-tuning stage.

Try these questions:

- **Training vs. held-out loss:** Did both fall? If only training loss fell, what might be happening?
- **Corpora:** Repeat with a different corpus at the same model size and step budget. Are the resulting loss curves even directly comparable when the text distribution differs?
- **Compute:** Hold the corpus constant, then compare tiny vs. 124M. Does more capacity *automatically* help under a tiny training budget?
- **Math:** Does memorizing question/answer text imply the model learned arithmetic? How would you design a test with new addition pairs?
- **Tokenization:** We reused GPT-2's tokenizer. What *else* would have to change to train literally everything from scratch?

If you want to build the components of a transformer yourself rather than train a provided architecture, [ARENA](https://arena.education/) takes you further.

Want to keep the weights? After a successful run, execute `scratch_model.save_pretrained("/content/hctigt-gpt2")` and `scratch_tokenizer.save_pretrained("/content/hctigt-gpt2")` in a new cell. These files live in the Colab runtime until you save them elsewhere.

**What this exercise does not establish:** that the model is good at language, generalizes, or matches historical GPT-2 training. A pretraining *run* is not a pretrained *result*.
